In [1]:
import tensorflow_model_analysis as tfma
import pandas as pd

2026-04-14 21:22:39.324372: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-14 21:22:39.464415: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /opt/ros/jazzy/opt/gz_sim_vendor/lib:/opt/ros/jazzy/opt/gz_sensors_vendor/lib:/opt/ros/jazzy/opt/gz_physics_vendor/lib:/opt/ros/jazzy/opt/sdformat_vendor/lib:/opt/ros/jazzy/opt/rviz_ogre_vendor/lib:/opt/ros/jazzy/lib/x86_64-linux-gnu:/opt/ros/jazzy/opt/gz_gui_vendor/lib:/opt/ros/jazzy/opt/gz_transport_vendor/lib:/opt/ros/jazzy/opt/gz_rendering_vendor/lib:/opt/ros/jazzy/opt/gz_plugin_vendor/lib:

In [2]:
eval_result_1= tfma.load_eval_result(
    '/home/manjula/COMP315/airflow_pipeline_outputs/adult_census_tfx_output/Evaluator/evaluation/42'
)

In [3]:
eval_result_2= tfma.load_eval_result(
    '/home/manjula/COMP315/airflow_pipeline_outputs/adult_census_tfx_output/Evaluator/evaluation/33'
)

In [4]:
import tensorflow_model_analysis as tfma

path_run_1 = './tfx_pipeline_output_v2/Evaluator/evaluation/8' 
path_run_2 = path_run_1 

eval_result_1 = tfma.load_eval_result(path_run_1)
eval_result_2 = tfma.load_eval_result(path_run_2)

metrics_1 = eval_result_1.get_metrics_for_all_slices()
metrics_2 = eval_result_2.get_metrics_for_all_slices()

for slice_key in metrics_1.keys():
    m1 = metrics_1[slice_key]
    m2 = metrics_2[slice_key]
    
    print(f"\nSlice: {slice_key}")
    acc1 = m1.get('binary_accuracy', m1.get('accuracy', 'N/A'))
    acc2 = m2.get('binary_accuracy', m2.get('accuracy', 'N/A'))
    
    print(f"  Run 1 accuracy: {acc1}")
    print(f"  Run 2 accuracy: {acc2}")

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`



Slice: (('race', 0),)
  Run 1 accuracy: {'doubleValue': 0.8252261841405003}
  Run 2 accuracy: {'doubleValue': 0.8252261841405003}

Slice: (('sex', 0),)
  Run 1 accuracy: {'doubleValue': 0.7943482963764197}
  Run 2 accuracy: {'doubleValue': 0.7943482963764197}

Slice: ()
  Run 1 accuracy: {'doubleValue': 0.8341180731618979}
  Run 2 accuracy: {'doubleValue': 0.8341180731618979}

Slice: (('sex', 1),)
  Run 1 accuracy: {'doubleValue': 0.9147478070175439}
  Run 2 accuracy: {'doubleValue': 0.9147478070175439}

Slice: (('race', 1),)
  Run 1 accuracy: {'doubleValue': 0.9030192131747484}
  Run 2 accuracy: {'doubleValue': 0.9030192131747484}

Slice: (('race', 2),)
  Run 1 accuracy: {'doubleValue': 0.8136986301369863}
  Run 2 accuracy: {'doubleValue': 0.8136986301369863}

Slice: (('race', 4),)
  Run 1 accuracy: {'doubleValue': 0.9130434782608695}
  Run 2 accuracy: {'doubleValue': 0.9130434782608695}

Slice: (('race', 3),)
  Run 1 accuracy: {'doubleValue': 0.9191919191919192}
  Run 2 accuracy: {'

In [5]:
import tensorflow_data_validation as tfdv
from tensorflow_metadata.proto.v0 import statistics_pb2

stats_path = './tfx_pipeline_output_v2/StatisticsGen/statistics/3/Split-train/FeatureStats.pb'
stats_proto = statistics_pb2.DatasetFeatureStatisticsList()

with open(stats_path, 'rb') as f:
    stats_proto.ParseFromString(f.read())

tfdv.visualize_statistics(stats_proto)

from tensorflow_metadata.proto.v0 import anomalies_pb2

anomalies_path = './tfx_pipeline_output_v2/ExampleValidator/anomalies/5/Split-train/SchemaDiff.pb'
anomalies_proto = anomalies_pb2.Anomalies()

with open(anomalies_path, 'rb') as f:
    anomalies_proto.ParseFromString(f.read())

tfdv.display_anomalies(anomalies_proto)


In [6]:
tfdv.visualize_statistics(stats_proto)
output_html = "data_stats.html"
with open(output_html, "w") as f:
    f.write(tfdv.utils.display_util.get_statistics_html(stats_proto))

print(f"Stats saved to {output_html}. Open this file in your browser to take a screenshot!")

Stats saved to data_stats.html. Open this file in your browser to take a screenshot!


In [8]:
import tensorflow as tf
from witwidget.notebook.visualization import WitConfigBuilder, WitWidget
import os

# Пути взяты напрямую из твоего вывода find
data_path = './tfx_pipeline_output_v2/CsvExampleGen/examples/1/Split-eval/data_tfrecord-00000-of-00001.gz'
model_path = './tfx_pipeline_output_v2/Pusher/pushed_model/9'

if os.path.exists(data_path) and os.path.exists(model_path):
    print(f"✅ Файлы подтверждены!")
    
    def parse_tfrecord(path):
        raw_dataset = tf.data.TFRecordDataset([path], compression_type="GZIP")
        examples = []
        for raw_record in raw_dataset.take(100):
            example = tf.train.Example()
            example.ParseFromString(raw_record.numpy())
            examples.append(example)
        return examples

    test_examples = parse_tfrecord(data_path)

    config_builder = (WitConfigBuilder(test_examples)
        .set_estimator_and_feature_spec(model_path, None)
        .set_target_feature('target') 
        .set_model_type('classification'))

    print("🚀 Запускаю виджет...")
    display(WitWidget(config_builder, height=800))
else:
    print("❌ Файлы не найдены. Проверь:")
    print(f"Данные существуют: {os.path.exists(data_path)}")
    print(f"Модель существует: {os.path.exists(model_path)}")

✅ Файлы подтверждены!
🚀 Запускаю виджет...


2026-04-14 21:23:22.043151: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:967] could not open file to read NUMA node: /sys/bus/pci/devices/0000:10:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-14 21:23:22.043455: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /opt/ros/jazzy/opt/gz_sim_vendor/lib:/opt/ros/jazzy/opt/gz_sensors_vendor/lib:/opt/ros/jazzy/opt/gz_physics_vendor/lib:/opt/ros/jazzy/opt/sdformat_vendor/lib:/opt/ros/jazzy/opt/rviz_ogre_vendor/lib:/opt/ros/jazzy/lib/x86_64-linux-gnu:/opt/ros/jazzy/opt/gz_gui_vendor/lib:/opt/ros/jazzy/opt/gz_transport_vendor/lib:/opt/ros/jazzy/opt/gz_rendering_vendor/lib:/opt/ros/jazzy/opt/gz_plugin_vendor/lib:/opt/ros/jazzy/opt/gz_fuel_tools_vendor/lib:/opt/ros/jazzy/opt/gz_msgs_vendor/lib:/opt/ros/jazzy/opt/gz_

WitWidget(config={'model_type': 'classification', 'label_vocab': [], 'are_sequence_examples': False, 'inferenc…